# Data Scrapping
Ce notebook récupère toutes les données brutes et les sauvegarde dans `data/`.

**Fichiers produits :**
- `data/games.csv` — Matchs (FBref)
- `data/managers.csv` — Managers (API + Transfermarkt)
- `data/meteo.csv` — Météo (Open-Meteo)
- `data/referees.csv` — Arbitres (API + Statshub)

**Ordre d'exécution recommandé :**
1. Setup
2. Games (FBref)
3. Stades (géocodage)
4. Météo
5. Managers (API)
6. Managers (Transfermarkt — historique)
7. Referees (API)
8. Referees manquants (Statshub)

## 1. Setup

In [1]:
import json
import os
import time

import pandas as pd
import requests
from dotenv import load_dotenv
from io import StringIO
from datetime import datetime

from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter

import undetected_chromedriver as uc
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.keys import Keys
from webdriver_manager.chrome import ChromeDriverManager
from bs4 import BeautifulSoup
from difflib import get_close_matches

load_dotenv()

BASE_URL = "https://sports.bzzoiro.com/api"
TOKEN = os.environ["BZZOIRO_TOKEN"]
HEADERS = {"Authorization": f"Token {TOKEN}"}

os.makedirs("data", exist_ok=True)
print("✅ Setup OK")

✅ Setup OK


## 2. Helpers API

In [2]:
def fetch_all(endpoint, params=None, verbose=True):
    """Pagination offset-based (limit/offset)."""
    params = dict(params or {})
    params["limit"] = 200
    results, offset = [], 0
    while True:
        params["offset"] = offset
        r = requests.get(f"{BASE_URL}/{endpoint}/", headers=HEADERS, params=params, timeout=30)
        r.raise_for_status()
        data = r.json()
        batch = data.get("results", [])
        results.extend(batch)
        if verbose:
            print(f"  {endpoint}: {len(results)}/{data.get('count', '?')}")
        if not data.get("next"):
            break
        offset += 200
        time.sleep(0.2)
    return results


def fetch_all_pages(endpoint, params=None, verbose=True):
    """Pagination page-based (page/page_size)."""
    params = dict(params or {})
    params["page_size"] = 500
    results, page = [], 1
    while True:
        params["page"] = page
        r = requests.get(f"{BASE_URL}/{endpoint}/", headers=HEADERS, params=params, timeout=30)
        r.raise_for_status()
        data = r.json()
        batch = data.get("results", [])
        results.extend(batch)
        if verbose:
            print(f"  {endpoint}: {len(results)}/{data.get('count', '?')}")
        if not data.get("next"):
            break
        page += 1
        time.sleep(0.2)
    return results

print("✅ Helpers API définis")

✅ Helpers API définis


## 3. Games — FBref

In [ ]:
SAISONS = ['2025-2026', '2024-2025', '2023-2024', '2022-2023', '2021-2022']
LEAGUE_MAP_FBREF = {
    "La-Liga": "12",
    "Bundesliga": "20",
    "Premier-League": "9",
    "Ligue-1": "13",
    "Serie-A": "11"
}

print("🚀 Lancement du navigateur FBref...")
options = uc.ChromeOptions()
# options.add_argument('--headless')  # Décommenter pour mode silencieux
driver = uc.Chrome(options=options, version_main=147)

results = pd.DataFrame()

try:
    for name, comp_id in LEAGUE_MAP_FBREF.items():
        for saison in SAISONS:
            URL = f"https://fbref.com/en/comps/{comp_id}/{saison}/schedule/{saison}-{name}-Scores-and-Fixtures"
            table_id = f"sched_{saison}_{comp_id}_1"
            print(f"\n--- {name} ({saison}) ---")

            try:
                driver.get(URL)

                # Gestion cookies
                try:
                    btn = WebDriverWait(driver, 3).until(
                        EC.element_to_be_clickable((By.CSS_SELECTOR, "button.osano-cm-accept-all"))
                    )
                    btn.click()
                    time.sleep(1)
                except Exception:
                    pass

                WebDriverWait(driver, 10).until(
                    EC.presence_of_element_located((By.ID, table_id))
                )
                html_source = driver.page_source

            except Exception as e:
                print(f"❌ Erreur chargement page : {e}")
                time.sleep(10)
                continue

            try:
                tables = pd.read_html(StringIO(html_source), attrs={'id': table_id})
                df = tables[0]
                df = df[['Date', 'Time', 'Home', 'Score', 'Away', 'Attendance', 'Venue', 'Referee']]
                df = df.dropna(subset=['Date'])
                df = df[df['Date'] != 'Date']
                df.insert(0, 'League', name.replace('-', ' '))
                df.insert(1, 'Season', saison)
                df = df.reset_index(drop=True)
                results = pd.concat([results, df], ignore_index=True)
                print(f"✅ {len(df)} matchs extraits")

            except ValueError:
                print(f"❌ Tableau {table_id} introuvable")
            except Exception as e:
                print(f"❌ Erreur extraction : {e}")

            print("⏳ Pause anti-ban (5s)...")
            time.sleep(5)

finally:
    driver.quit()
    print("\n🛑 Navigateur fermé")

# Nettoyage : on ne garde que les matchs joués (Score renseigné)
results = results.dropna(subset=["Score"])
results = results[results['Date'] <= "2026-04-30"]
results.to_csv("../../data/foot/games.csv", index=False)
print(f"\n🎉 {len(results)} matchs sauvegardés dans data/games.csv")

🚀 Lancement du navigateur FBref...

--- La-Liga (2025-2026) ---
✅ 380 matchs extraits
⏳ Pause anti-ban (5s)...

--- La-Liga (2024-2025) ---
✅ 380 matchs extraits
⏳ Pause anti-ban (5s)...

--- La-Liga (2023-2024) ---
✅ 380 matchs extraits
⏳ Pause anti-ban (5s)...

--- La-Liga (2022-2023) ---
✅ 380 matchs extraits
⏳ Pause anti-ban (5s)...

--- La-Liga (2021-2022) ---
✅ 380 matchs extraits
⏳ Pause anti-ban (5s)...

--- Bundesliga (2025-2026) ---
✅ 306 matchs extraits
⏳ Pause anti-ban (5s)...

--- Bundesliga (2024-2025) ---
✅ 306 matchs extraits
⏳ Pause anti-ban (5s)...

--- Bundesliga (2023-2024) ---
✅ 306 matchs extraits
⏳ Pause anti-ban (5s)...

--- Bundesliga (2022-2023) ---
✅ 306 matchs extraits
⏳ Pause anti-ban (5s)...

--- Bundesliga (2021-2022) ---
✅ 306 matchs extraits
⏳ Pause anti-ban (5s)...

--- Premier-League (2025-2026) ---
✅ 380 matchs extraits
⏳ Pause anti-ban (5s)...

--- Premier-League (2024-2025) ---
✅ 380 matchs extraits
⏳ Pause anti-ban (5s)...

--- Premier-League (202

## 4. Stades — Géocodage

In [10]:
# Correspondance ligue → pays
MAP_PAYS = {
    "La Liga": "Spain",
    "Bundesliga": "Germany",
    "Premier League": "United Kingdom",
    "Ligue 1": "France",
    "Serie A": "Italy"
}

# Corrections manuelles pour stades introuvables via geocoder
STADES_FIXES = {
    'Estadio de Mestalla': (39.4746, -0.3582),
    'Estadio Nuevo Carlos Tartiere': (43.3603, -5.8711),
    'Estadio Municipal José Zorrilla': (41.6447, -4.7612),
    'Riyadh Air Metropolitan Stadium': (40.4362, -3.5995),
    'Estadio Cívitas Metropolitano': (40.4362, -3.5995),
    'Estadio Wanda Metropolitano': (40.4362, -3.5995),
    'Estadio del Rayo Vallecano': (40.3917, -3.6587),
    'Iberostar Estadi': (39.5900, 2.6300),
    'Estadio Municipal de Anoeta': (43.3014, -1.9736),
    'Ursapharm-Arena an der Kaiserlinde': (49.3186, 7.1228),
    'Merkur Spielarena': (51.2616, 6.7332),
    'BWT-Stadion Am Hardtwald': (49.3330, 8.5794),
    'Wohninvest-Weserstadion': (53.0664, 8.8376),
    'The American Express Community Stadium': (50.8618, -0.0837),
    "Stade de l'Abbé Deschamps": (47.7865, 3.5883),
    'Stade de la Mosson-Mondial 98': (43.6217, 3.8122),
    'Stade Matmut-Atlantique': (44.8974, -0.5615),
    "Stade de l'Aube (Neutral Site)": (48.3072, 4.0954),
    'Stadio Comunale Ettore Giardiniero': (40.3614, 18.2086),
    'Arena Garibaldi - Stadio Romeo Anconetan...': (43.7240, 10.4024),
    "Stadio Marc'Antonio Bentegodi": (45.4353, 10.9686),
    'Stadio Carlo Castellani - Computer Gross...': (43.7264, 10.9550),
    'Gewiss Stadium': (45.7091, 9.6808),
    'Stadio Comunale Via Del Mare': (40.3614, 18.2086),
    'Wohninvest Weserstadion' : (53.0598364273, 8.83590665636),
    'Schwarzwald-Stadion' : (47.9959, 7.85222)
}

print("✅ Config stades chargée")

✅ Config stades chargée


In [ ]:
# Récupération des villes depuis l'API (pl_events optionnel)
# Si vous avez déjà pl_events.csv, cette cellule peut être sautée
LEAGUE_MAP_API = {
    "premier league": [337, 336, 335, 334, 333],
    "la liga": [294, 293, 292, 291, 290],
    "ligue 1": [317, 316, 315, 314, 313],
    "bundesliga": [228, 227, 226, 225, 224],
    "serie A": [358, 357, 356, 355, 354]
}

def flatten_event(ev):
    row = dict(ev)
    for key, fields in [
        ("league",        ["id", "name", "slug"]),
        ("season",        ["id", "name", "year"]),
        ("home_team_obj", ["id", "name", "short_name"]),
        ("away_team_obj", ["id", "name", "short_name"]),
        ("home_coach",    ["id", "name"]),
        ("away_coach",    ["id", "name"]),
        ("referee",       ["id", "name"]),
        ("venue",         ["id", "name", "city", "capacity"]),
    ]:
        obj = row.pop(key, None) or {}
        for f in fields:
            row[f"{key}_{f}"] = obj.get(f)

    for key in ("unavailable_players", "lineups", "shotmap", "momentum"):
        val = row.get(key)
        row[key] = json.dumps(val, ensure_ascii=False) if val is not None else None

    odds = row.pop("odds", None) or {}
    row["odds_home"] = odds.get("home")
    row["odds_draw"] = odds.get("draw")
    row["odds_away"] = odds.get("away")
    return row

all_events = []
for name, seasons in LEAGUE_MAP_API.items():
    for season in seasons:
        print(f"Season {season}...")
        all_events += fetch_all("events", {"season": season, "full": "true"})

pl_events = pd.DataFrame([flatten_event(ev) for ev in all_events])
pl_events.to_csv("../../data/foot/pl_events.csv", index=False)
print(f"\n✅ pl_events: {pl_events.shape}")

Season 337...
  events: 200/385
  events: 385/385
Season 336...
  events: 200/380
  events: 380/380
Season 335...
  events: 200/380
  events: 380/380
Season 334...
  events: 200/380
  events: 380/380
Season 333...
  events: 200/380
  events: 380/380
Season 294...
  events: 200/383
  events: 383/383
Season 293...
  events: 200/380
  events: 380/380
Season 292...
  events: 200/380
  events: 380/380
Season 291...
  events: 200/380
  events: 380/380
Season 290...
  events: 200/380
  events: 380/380
Season 317...
  events: 200/309
  events: 309/309
Season 316...
  events: 200/306
  events: 306/306
Season 315...
  events: 200/306
  events: 306/306
Season 314...
  events: 200/380
  events: 380/380
Season 313...
  events: 200/380
  events: 380/380
Season 228...
  events: 200/309
  events: 309/309
Season 227...
  events: 200/306
  events: 306/306
Season 226...
  events: 200/308
  events: 308/308
Season 225...
  events: 200/306
  events: 306/306
Season 224...
  events: 200/306
  events: 306/306


In [ ]:
# Geocodage des stades
geolocator = Nominatim(user_agent="football_weather_pipeline_v2", timeout=10)
cache_coords = {}

def obtenir_coordonnees(venue, city, country):
    if venue in cache_coords:
        return cache_coords[venue]

    # Correction manuelle prioritaire
    if venue in STADES_FIXES:
        cache_coords[venue] = STADES_FIXES[venue]
        return STADES_FIXES[venue]

    try:
        location = geolocator.geocode(f"{venue}, {city}, {country}")
        if location is None:
            location = geolocator.geocode(f"{city}, {country}")
        if location:
            res = (location.latitude, location.longitude)
        else:
            print(f"[GPS] Introuvable : {venue}")
            res = (None, None)
        cache_coords[venue] = res
        time.sleep(1.2)
        return res
    except Exception as e:
        print(f"[GPS] Erreur réseau pour {venue} : {e}")
        return (None, None)


df_matchs = pd.read_csv("../../data/foot/games.csv")
venues_api = pd.read_csv("../../data/foot/pl_events.csv")[["venue_name", "venue_city"]].drop_duplicates()
venues_dict = venues_api.set_index("venue_name")["venue_city"].to_dict()

historique_stades = []
for _, row in df_matchs.iterrows():
    venue = row["Venue"]
    city = venues_dict.get(venue, venue)
    country = MAP_PAYS[row["League"]]
    lat, lon = obtenir_coordonnees(venue, city, country)
    historique_stades.append({"name": venue, "city": city, "country": country, "lat": lat, "lon": lon})

df_stades = pd.DataFrame(historique_stades).drop_duplicates()
df_stades.to_csv("../../data/foot/stades.csv", index=False)
print(f"✅ {len(df_stades)} stades sauvegardés dans stades.csv")

# Vérification des stades sans coordonnées
manquants = df_stades[df_stades["lat"].isna()]["name"].unique()
if len(manquants) > 0:
    print(f"⚠️ {len(manquants)} stades sans coordonnées : {manquants}")
else:
    print("✅ Tous les stades ont des coordonnées")

C:\Users\MUEL\AppData\Local\Temp\ipykernel_9620\1006454429.py:32: DtypeWarning: Columns (0: period, 1: is_local_derby, 2: incidents, 3: live_stats, 4: lineups, 5: shotmap, 6: momentum, 7: xg_per_minute, 8: average_positions, 9: jerseys, 10: funfacts, 11: sr_stats, 12: home_coach_name, 13: away_coach_name, 14: referee_name) have mixed types. Specify dtype option on import or set low_memory=False.
  venues_api = pd.read_csv("data/pl_events.csv")[["venue_name", "venue_city"]].drop_duplicates()


[GPS] Erreur réseau pour Estadio Benito Villamarín : Non-successful status code 429
[GPS] Erreur réseau pour Estadio de Gran Canaria : Non-successful status code 429
[GPS] Erreur réseau pour Estadio de Gran Canaria : Non-successful status code 429


KeyboardInterrupt: 

## 5. Météo — Open-Meteo

In [12]:
def obtenir_meteo_match(club_dom, club_ext, lat, lon, date_match, time_match):
    """Récupère les variables météo pour la durée d'un match."""
    if pd.isna(lat) or pd.isna(lon):
        return {}

    params = {
        "latitude": lat,
        "longitude": lon,
        "start_date": date_match,
        "end_date": date_match,
        "hourly": "temperature_2m,relative_humidity_2m,precipitation,snowfall,wind_speed_10m,wind_gusts_10m",
        "timezone": "Europe/London"
    }

    response = requests.get("https://archive-api.open-meteo.com/v1/archive", params=params)

    if response.status_code != 200:
        print(f"[Météo] Erreur API le {date_match}: {response.text}")
        return {}

    data = response.json()
    df_meteo = pd.DataFrame(data["hourly"])
    df_meteo["time"] = pd.to_datetime(df_meteo["time"])

    kickoff = pd.to_datetime(f"{date_match} {time_match}")
    full_time = kickoff + pd.Timedelta(hours=2)
    df_match = df_meteo[(df_meteo["time"] >= kickoff) & (df_meteo["time"] <= full_time)]

    if df_match.empty:
        return {}

    return {
        "Home_Team": club_dom,
        "Away_Team": club_ext,
        "Date": date_match,
        "Temp_Moy_C": round(df_match["temperature_2m"].mean(), 1),
        "Humidite_Moy_%": round(df_match["relative_humidity_2m"].mean(), 1),
        "Pluie_Tot_mm": round(df_match["precipitation"].sum(), 1),
        "Neige_Tot_cm": round(df_match["snowfall"].sum(), 1),
        "Vent_Moy_kmh": round(df_match["wind_speed_10m"].mean(), 1),
        "Rafale_Max_kmh": round(df_match["wind_gusts_10m"].max(), 1),
    }

print("✅ Fonction météo définie")

✅ Fonction météo définie


In [ ]:
df_matchs = pd.read_csv("../../data/foot/games.csv")
df_stades = pd.read_csv("../../data/foot/stades.csv")

map_lat = df_stades.set_index("name")["lat"].to_dict()
map_lon = df_stades.set_index("name")["lon"].to_dict()

# Reprise incrémentale : on ne scrape pas les matchs déjà présents
meteo_existante_path = "../../data/foot/meteo.csv"
if os.path.exists(meteo_existante_path):
    df_meteo_existante = pd.read_csv(meteo_existante_path)
    deja_presents = set(
        zip(df_meteo_existante["Home_Team"], df_meteo_existante["Away_Team"], df_meteo_existante["Date"])
    )
    print(f"♻️ {len(deja_presents)} matchs météo déjà présents, reprise incrémentale...")
else:
    df_meteo_existante = pd.DataFrame()
    deja_presents = set()

historique_meteo = []
for _, row in df_matchs.iterrows():
    venue = row["Venue"]
    home, away, date = row["Home"], row["Away"], row["Date"]

    if (home, away, date) in deja_presents:
        continue

    heure = str(row["Time"]).split(" ")[0]
    lat = map_lat.get(venue)
    lon = map_lon.get(venue)
    meteo = obtenir_meteo_match(home, away, lat, lon, date, heure)
    historique_meteo.append(meteo)

df_meteo_nouvelle = pd.DataFrame(historique_meteo).dropna(how="all")
df_meteo = pd.concat([df_meteo_existante, df_meteo_nouvelle], ignore_index=True)
df_meteo.to_csv(meteo_existante_path, index=False)
print(f"✅ {len(df_meteo)} entrées météo sauvegardées dans data/meteo.csv")

♻️ 8721 matchs météo déjà présents, reprise incrémentale...
✅ 10225 entrées météo sauvegardées dans data/meteo.csv


## 6. Managers — API Bzzoiro

In [ ]:
def flatten_manager(m):
    row = dict(m)
    team = row.pop("current_team", None) or {}
    row["current_team_id"] = team.get("id")
    row["current_team_name"] = team.get("name")
    row["tactical_styles"] = json.dumps(row.get("tactical_styles", []), ensure_ascii=False)
    return row

managers_raw = fetch_all_pages("managers")
managers = pd.DataFrame([flatten_manager(m) for m in managers_raw])
managers.to_csv("../../data/foot/managers.csv", index=False)
print(f"✅ {managers.shape[0]} managers sauvegardés dans data/managers.csv")

  managers: 50/1439
  managers: 100/1439
  managers: 150/1439
  managers: 200/1439
  managers: 250/1439
  managers: 300/1439
  managers: 350/1439
  managers: 400/1439
  managers: 450/1439
  managers: 500/1439
  managers: 550/1439
  managers: 600/1439
  managers: 650/1439
  managers: 700/1439
  managers: 750/1439
  managers: 800/1439
  managers: 850/1439
  managers: 900/1439
  managers: 950/1439
  managers: 1000/1439
  managers: 1050/1439
  managers: 1100/1439
  managers: 1150/1439
  managers: 1200/1439
  managers: 1250/1439
  managers: 1300/1439
  managers: 1350/1439
  managers: 1400/1439
  managers: 1439/1439
✅ 1439 managers sauvegardés dans data/managers.csv


## 7. Managers — Historique Transfermarkt

In [ ]:
# Mapping équipe → ID Transfermarkt
TEAM_ID_MAP = {
    'Ajaccio': 1147, 'Alavés': 1108, 'Almería': 3302, 'Angers': 1420,
    'Arminia': 10, 'Arsenal': 11, 'Aston Villa': 405, 'Atalanta': 800,
    'Athletic Club': 621, 'Atlético Madrid': 13, 'Augsburg' : 167, 'Auxerre': 290,
    'Barcelona': 131, 'Leverkusen' : 15, 'Bayern Munich' : 27, 'Bochum': 80, 'Bologna': 1025, 'Bordeaux': 40,
    'Dortmund' : 16, 'Gladbach' : 18, 'Bournemouth': 989, 'Brentford': 1148, 'Brest': 3911, 'Brighton': 1237,
    'BTSV': 23, 'Burnley': 1132, 'Cádiz': 2687, 'Cagliari': 1390,
    'Celta Vigo': 940, 'Chelsea': 631, 'Clermont Foot': 3524, 'Como': 1047,
    'Cremonese': 2239, 'Crystal Palace': 873, 'Darmstadt 98': 105,
    'Dresden': 129, 'Düsseldorf': 38, 'Eintracht Frankfurt' : 24, 'Elche': 1531, 'Elversberg': 64,
    'Empoli': 749, 'Erzgebirge Aue': 94, 'Espanyol': 714, 'Everton': 29,
    'Fiorentina': 430, 'Freiburg' : 60, 'Frosinone': 8970, 'Fulham': 931, 'Genoa': 252,
    'Getafe': 3709, 'Girona': 12321, 'Granada': 16795, 'Greuther Fürth': 65,
    'Hamburger SV': 41, 'Hannover 96': 42, 'Hansa Rostock': 30,
    'Heidenheim': 2036, 'Hellas Verona': 276, 'Hertha BSC': 44,
    'Hoffenheim' : 533, 'Holstein Kiel': 269, 'Ingolstadt': 4795, 'Inter': 46, 'Ipswich Town': 677,
    'Jahn Regensburg': 109, 'Juventus': 506, 'Kaiserslautern': 2,
    'Karlsruher': 48, 'Köln': 3, 'Las Palmas': 472, 'Lazio': 398,
    'Le Havre': 738, 'Lecce': 1005, 'Leeds United': 399, 'Leganés': 1244,
    'Leicester City': 1003, 'Lens': 826, 'Levante': 3368, 'Lille': 1082,
    'Liverpool': 31, 'Lorient': 1158, 'Luton Town': 1031, 'Lyon': 1041,
    'Magdeburg': 187, 'Mainz 05' : 39, 'Mallorca': 237, 'Manchester City': 281,
    'Manchester Utd': 985, 'Marseille': 244, 'Metz': 347, 'Milan': 5,
    'Monaco': 162, 'Montpellier': 969, 'Monza': 2919, 'Nantes': 995,
    'Napoli': 6195, 'Newcastle United': 762, 'Nice': 417, 'Nürnberg': 4,
    'Osasuna': 331, 'Osnabrück': 81, 'Oviedo': 2497, 'Paderborn 07': 127,
    'Paris FC': 10004, 'Paris Saint-Germain': 583, 'Parma': 130, 'Pisa': 4172,
    'Preußen Münster': 91, 'Rayo Vallecano': 367, 'RB Leipzig' : 23826, 'Real Betis': 150,
    'Real Madrid': 418, 'Real Sociedad': 681, 'Reims': 1421, 'Rennes': 273,
    'Roma': 12, 'Saint-Étienne': 618, 'Salernitana': 380, 'Sampdoria': 1038,
    'Sandhausen': 254, 'Sassuolo': 6574, 'Schalke 04': 33, 'Sevilla': 368,
    'Sheffield United': 350, 'Southampton': 180, 'Spezia': 3522, 'St Pauli': 35,
    'Strasbourg': 667, 'Sunderland': 289, 'Torino': 416,
    'Tottenham Hotspur': 148, 'Toulouse': 415, 'Troyes': 1095, 'Udinese': 410,
    'Ulm': 69, 'Union Berlin' : 89, 'Valencia': 1049, 'Valladolid': 366, 'Venezia': 607,
    'Stuttgart' : 79, 'Villarreal': 1050, 'Watford': 1010, 'Wehen Wiesbaden': 108,
    'Werder Bremen': 86, 'West Ham United': 379, 'Wolves': 543, 'Wolfsburg' : 82,
    'Nottingham Forest': 703, 'Norwich City': 1123
}

# Vérification : équipes dans games.csv non couvertes par le mapping
df_matchs = pd.read_csv("../../data/foot/games.csv")
teams_in_games = set(df_matchs["Home"].unique()) | set(df_matchs["Away"].unique())
teams_missing = teams_in_games - set(TEAM_ID_MAP.keys())
if teams_missing:
    print(f"⚠️ {len(teams_missing)} équipes sans ID Transfermarkt : {teams_missing}")
else:
    print("✅ Toutes les équipes sont dans le mapping")

✅ Toutes les équipes sont dans le mapping


In [16]:
MOIS_FR = {
    'janv.': 'Jan', 'févr.': 'Feb', 'mars': 'Mar', 'avr.': 'Apr',
    'mai': 'May', 'juin': 'Jun', 'juil.': 'Jul', 'août': 'Aug',
    'sept.': 'Sep', 'oct.': 'Oct', 'nov.': 'Nov', 'déc.': 'Dec'
}
TODAY_STR = datetime.now().strftime('%Y-%m-%d')


def clean_tm_date(date_str, is_departure=False):
    """Convertit une date Transfermarkt (français) en YYYY-MM-DD."""
    if pd.isna(date_str) or str(date_str).strip() in ["", "-", "nan"]:
        return TODAY_STR if is_departure else None
    new_date = str(date_str).lower()
    for fr, en in MOIS_FR.items():
        if fr in new_date:
            new_date = new_date.replace(fr, en)
            break
    try:
        return pd.to_datetime(new_date, dayfirst=True).strftime('%Y-%m-%d')
    except Exception:
        return TODAY_STR if is_departure else None


def scrape_managers_transfermarkt(driver, club_name, club_id):
    """Scrape l'historique des managers d'un club sur Transfermarkt."""
    url = f"https://www.transfermarkt.fr/test/mitarbeiterhistorie/verein/{club_id}"
    print(f"🔍 {club_name} (ID: {club_id})...")
    driver.get(url)
    time.sleep(3)

    soup = BeautifulSoup(driver.page_source, 'html.parser')
    table = soup.find('table', class_='items')
    if not table:
        print(f"❌ Tableau introuvable pour {club_name}")
        return None

    rows = []
    tbody = table.find('tbody')
    if tbody:
        for tr in tbody.find_all('tr', class_=['odd', 'even']):
            row_data = {'Club': club_name}
            tds = tr.find_all('td', recursive=False)
            if len(tds) < 6:
                continue
            name_td = tds[0].find('td', class_='hauptlink')
            if name_td and name_td.find('a'):
                row_data['Manager'] = name_td.find('a').text.strip()
            else:
                continue
            row_data['Nommé'] = tds[2].text.strip()
            row_data['Départ'] = tds[3].text.strip()
            row_data['PPM'] = tds[6].text.strip()
            rows.append(row_data)

    return pd.DataFrame(rows) if rows else None

print("✅ Fonctions Transfermarkt définies")

✅ Fonctions Transfermarkt définies


In [ ]:
tm_managers_history = pd.read_csv('../../data/foot/transfermarkt_managers_history.csv')
tm_clubs = tm_managers_history['Club'].unique().tolist()

In [ ]:
chrome_options = Options()
# chrome_options.add_argument("--headless")
chrome_options.add_argument("--no-sandbox")
chrome_options.add_argument("--disable-dev-shm-usage")
chrome_options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36")

print("🚀 Lancement du navigateur Transfermarkt...")
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=chrome_options)

all_managers_dfs = []
try:
    for team, club_id in TEAM_ID_MAP.items():
        if team in tm_clubs :
            pass
        else :
            df_club = scrape_managers_transfermarkt(driver, team, club_id)
            if df_club is not None and not df_club.empty:
                all_managers_dfs.append(df_club)
                print(f"  ✅ {len(df_club)} managers pour {team}")
except Exception as e:
    print(f"❌ Erreur critique : {e}")
finally:
    driver.quit()
    print("🛑 Navigateur fermé")

tm_managers = pd.concat(all_managers_dfs, ignore_index=True)

# Nettoyage des dates
tm_managers['Nommé'] = tm_managers['Nommé'].apply(lambda x: clean_tm_date(x, is_departure=False))
tm_managers['Départ'] = tm_managers['Départ'].apply(lambda x: clean_tm_date(x, is_departure=True))

tm_managers = pd.concat([tm_managers_history, tm_managers], ignore_index=True)
tm_managers.to_csv('../../data/foot/transfermarkt_managers_history.csv', index=False, encoding='utf-8-sig')
print(f"\n✅ {len(tm_managers)} entrées sauvegardées dans data/foot/transfermarkt_managers_history.csv")

🚀 Lancement du navigateur Transfermarkt...
🔍 Augsburg (ID: 167)...
  ✅ 64 managers pour Augsburg
🔍 Leverkusen (ID: 15)...
  ✅ 56 managers pour Leverkusen
🔍 Bayern Munich (ID: 27)...
  ✅ 68 managers pour Bayern Munich
🔍 Dortmund (ID: 16)...
  ✅ 67 managers pour Dortmund
🔍 Gladbach (ID: 18)...
  ✅ 44 managers pour Gladbach
🔍 Eintracht Frankfurt (ID: 24)...
  ✅ 68 managers pour Eintracht Frankfurt
🔍 Freiburg (ID: 60)...
  ✅ 29 managers pour Freiburg
🔍 Hoffenheim (ID: 533)...
  ✅ 24 managers pour Hoffenheim
🔍 Mainz 05 (ID: 39)...
  ✅ 57 managers pour Mainz 05
🔍 RB Leipzig (ID: 23826)...
  ✅ 15 managers pour RB Leipzig
🔍 Union Berlin (ID: 89)...
  ✅ 53 managers pour Union Berlin
🔍 Stuttgart (ID: 79)...
  ✅ 71 managers pour Stuttgart
🔍 Wolfsburg (ID: 82)...
  ✅ 55 managers pour Wolfsburg
🛑 Navigateur fermé

✅ 8934 entrées sauvegardées dans data/transfermarkt_managers_history.csv


## 8. Referees — API Bzzoiro

In [ ]:
referees_raw = fetch_all("referees")
referees = pd.DataFrame(referees_raw)
referees.to_csv("../../data/foot/referees.csv", index=False)
print(f"✅ {referees.shape[0]} arbitres sauvegardés dans data/foot/referees.csv")

  referees: 1354/1354
✅ 1354 arbitres sauvegardés dans data/referees.csv


## 9. Referees manquants — Statshub

In [24]:
# Mapping des noms FBref → noms API
MAP_REFEREES = {
    'Clément Turpin': 'Clement Turpin',
    'Matthias Jöllenbeck': 'Matthias Jollenbeck',
    'Florian Badstübner': 'Florian Badstubner',
    'Gaël Angoula': 'Gael Angoula',
    'Javier Alberola': 'Javier Alberola Rojas',
    'Ricardo de Burgos': 'Ricardo De Burgos Bengoetxea',
    'Adrián Cordero': 'Adrian Cordero Vega',
    'Francisco Hernández': 'Francisco Hernandez Maeso',
    'José Sánchez': 'José María Sánchez Martínez',
    'Iosu Galech': 'Iosu Galech Apezteguia',
    'Víctor García': 'Victor Garcia Verdura',
    'Ivan Pezzuto': 'Ivano Pezzuto',
    'José Guzmán': 'Jose Luis Guzman Mansilla',
    'Alejandro Muñíz': 'Alejandro Muñiz Ruiz',
    'Miguel Sesma': 'Miguel Sesma Espinosa',
    'Mateo Busquets': 'Mateo Busquets Ferrer',
    'César Soto': 'César Soto Grado',
    'Jesús Gil': 'Jesus Gil Manzano',
    'Guillermo Cuadra': 'Guillermo Cuadra Fernandez',
    'José Luis Munuera': 'José Luis Munuera Montero',
    'Miguel Ángel Ortiz Arias': 'Miguel Angel Ortiz Arias',
    'Alejandro Quintero': 'Alejandro Quintero Gonzalez',
    'Isidro Díaz de Mera': 'Isidro Diaz de Mera Escuderos',
    'Mario Melero': 'Mario Melero López',
    'Antonio Matéu Lahoz': 'Antonio Miguel Mateu Lahoz',
    'Jose Maria Sánchez': 'Jose Maria Sanchez Santos',
    'Juan Martínez': 'Juan Martinez Munuera',
    'Carlos del Cerro': 'Carlos del Cerro Grande',
    'Santiago Jaime': 'Santiago Jaime Latre',
    'Marc-Philipp Eckermann': 'Marc Philip Eckermann',
    'Samuel Allison': 'Sam Allison',
    'Jérémie Pignard': 'Jeremie Pignard',
    'Stéphanie Frappart': 'Stephanie Frappart',
    'Alberto Arena': 'Alberto Ruben Arena',
    'Juan Sacchi': 'Juan Luca Sacchi',
    'Alejandro Hernández': 'Alejandro Hernandez',
    'Darren Bond': 'Darren England',
    'Joshua Smith': 'Josh Smith',
    'António Nobre': 'Antonio Nobre',
    'Antonio Matéu': 'Antonio Miguel Mateu Lahoz',
    'Christian Ballweg': 'Cristian Ballweg',
    'Thomas Léonard': 'Thomas Leonard',
    'Javier Villanueva': 'Ignacio Iglesias Villanueva',
    'Víctor Acosta': 'Victor Acosta',
    'Pablo González': 'Pablo Gonzalez',
    'Valentín Pizarro': 'Valentin Pizarro',
    'Hsu Jason': 'Jason Hsu',
    'Christof Günsch': 'Christof Gunsch',
    'Niccolò Baroni': 'Niccolo Baroni',
    'Cosso Cosso': 'Francesco Cosso',
}


def get_avg_goals_referee(ref_name, df_games):
    """Calcule la moyenne de buts par match pour un arbitre."""
    scores = df_games.loc[df_games["Referee"] == ref_name, "Score"]
    if len(scores) == 0:
        return 0.0
    total = 0
    for score in scores:
        score_clean = str(score).replace('–', '-').replace('—', '-')
        parts = score_clean.split('-')
        if len(parts) == 2:
            try:
                total += int(parts[0].strip()) + int(parts[1].strip())
            except ValueError:
                pass
    return round(total / len(scores), 2)


def scrape_referees_statshub(arbitres_list, referees_df, df_games):
    """Scrape les stats des arbitres manquants sur Statshub."""
    print(f"🔄 Scraping de {len(arbitres_list)} arbitres sur Statshub...")

    chrome_options = Options()
    chrome_options.add_argument("--no-sandbox")
    chrome_options.add_argument("--disable-dev-shm-usage")
    chrome_options.add_argument("window-size=1920,1080")
    chrome_options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36")

    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=chrome_options)
    wait = WebDriverWait(driver, 10)

    try:
        driver.get("https://www.statshub.com/referees")

        # Désactivation du filtre "Upcoming only"
        try:
            toggle = wait.until(EC.presence_of_element_located((By.ID, "upcoming-fixtures-only")))
            if toggle.get_attribute("aria-checked") == "true" or toggle.get_attribute("data-state") == "checked":
                driver.execute_script("arguments[0].click();", toggle)
                time.sleep(1)
        except Exception:
            pass

        search_input = wait.until(EC.presence_of_element_located(
            (By.CSS_SELECTOR, "input[placeholder='Search referees...']"))
        )

        for arbitre in arbitres_list:
            print(f"\n🔍 Recherche de '{arbitre}'...")
            try:
                search_input.send_keys(Keys.CONTROL + "a")
                search_input.send_keys(Keys.DELETE)
                search_input.send_keys(arbitre)
                time.sleep(3)

                html = driver.page_source
                if "No results" in html or "No referees found" in html:
                    print(f"⏭️ Introuvable sur Statshub")
                    continue

                soup = BeautifulSoup(html, 'html.parser')
                table = soup.find('table')
                if not table:
                    continue

                headers = [th.text.strip() for th in table.find('thead').find_all('th')]
                rows = []
                tbody = table.find('tbody')
                if tbody:
                    for tr in tbody.find_all('tr'):
                        tds = tr.find_all('td')
                        if not tds:
                            continue
                        name_span = tds[0].find('span', class_='block')
                        full_name = name_span.text.strip() if name_span else tds[0].text.strip()
                        row_data = [full_name] + [td.text.strip() for td in tds[1:]]
                        rows.append(row_data)

                if rows:
                    temp_df = pd.DataFrame(rows, columns=headers)
                    if not temp_df.empty and "Yel PG" in temp_df.columns:
                        def safe_float(val):
                            return 0.0 if val == "—" else float(val)
                        referees_df.loc[len(referees_df)] = {
                            'name': arbitre,
                            'avg_yellow_per_match': safe_float(temp_df["Yel PG"].iloc[0]),
                            'avg_red_per_match': safe_float(temp_df["Red PG"].iloc[0]),
                            'avg_goals_per_match': get_avg_goals_referee(arbitre, df_games),
                            'avg_fouls_per_match': safe_float(temp_df["Fouls"].iloc[0]),
                        }
                        print(f"✅ {arbitre} ajouté")

            except Exception as e:
                print(f"⚠️ Erreur pour '{arbitre}' : {e}")
                continue

    except Exception as e:
        print(f"❌ Erreur critique : {e}")
    finally:
        driver.quit()
        print("🛑 Navigateur fermé")

    return referees_df

print("✅ Fonctions arbitres définies")

✅ Fonctions arbitres définies


In [ ]:
df_matchs = pd.read_csv("../../data/foot/games.csv")
referees = pd.read_csv("../../data/foot/referees.csv")

# Application du mapping de noms
df_matchs["Referee"] = df_matchs["Referee"].replace(MAP_REFEREES)

# Colonnes utiles uniquement
COLS_REFEREES_A_GARDER = [
    "name", "avg_yellow_per_match", "avg_red_per_match",
    "avg_goals_per_match", "avg_fouls_per_match"
]
referees_cols = [c for c in COLS_REFEREES_A_GARDER if c in referees.columns]
referees = referees[referees_cols]

# Identification des arbitres manquants
arbitres_manquants = df_matchs[~df_matchs["Referee"].isin(referees["name"])]["Referee"].dropna().unique()
print(f"⚠️ {len(arbitres_manquants)} arbitres manquants dans le référentiel")

# Scraping Statshub pour les manquants
if len(arbitres_manquants) > 0:
    referees = scrape_referees_statshub(arbitres_manquants, referees, df_matchs)

# Vérification finale
encore_manquants = df_matchs[~df_matchs["Referee"].isin(referees["name"])]["Referee"].dropna().unique()
if len(encore_manquants) > 0:
    print(f"⚠️ {len(encore_manquants)} arbitres toujours sans stats : {encore_manquants}")
else:
    print("✅ Tous les arbitres sont couverts")

# Sauvegarde games avec mapping appliqué + referees complet
df_matchs.to_csv("../../data/foot/games.csv", index=False)
referees.to_csv("../../data/foot/referees.csv", index=False)
print(f"\n✅ {len(referees)} arbitres sauvegardés dans data/foot/referees.csv")
print(f"✅ data/foot/games.csv mis à jour avec les noms d'arbitres normalisés")

⚠️ 39 arbitres manquants dans le référentiel
🔄 Scraping de 39 arbitres sur Statshub...

🔍 Recherche de 'Victor Acosta'...
⏭️ Introuvable sur Statshub

🔍 Recherche de 'Juan Pulido'...
⏭️ Introuvable sur Statshub

🔍 Recherche de 'Pablo Gonzalez'...
✅ Pablo Gonzalez ajouté

🔍 Recherche de 'Jorge Figueroa'...
✅ Jorge Figueroa ajouté

🔍 Recherche de 'Valentin Pizarro'...
✅ Valentin Pizarro ajouté

🔍 Recherche de 'Jason Hsu'...
⏭️ Introuvable sur Statshub

🔍 Recherche de 'Benjamin Cortus'...
✅ Benjamin Cortus ajouté

🔍 Recherche de 'Robert Schörgenhofer'...
⏭️ Introuvable sur Statshub

🔍 Recherche de 'David Coote'...
✅ David Coote ajouté

🔍 Recherche de 'Graham Scott'...
✅ Graham Scott ajouté

🔍 Recherche de 'Rebecca Welch'...
✅ Rebecca Welch ajouté

🔍 Recherche de 'Sunny Singh'...
⏭️ Introuvable sur Statshub

🔍 Recherche de 'Andre Marriner'...
✅ Andre Marriner ajouté

🔍 Recherche de 'Jonathan Moss'...
✅ Jonathan Moss ajouté

🔍 Recherche de 'Mike Dean'...
✅ Mike Dean ajouté

🔍 Recherche de '

## ✅ Récapitulatif des fichiers produits

In [26]:
fichiers = {
    "data/games.csv": "Matchs joués (FBref) avec noms arbitres normalisés",
    "data/managers.csv": "Managers (API Bzzoiro)",
    "data/meteo.csv": "Météo par match (Open-Meteo)",
    "data/referees.csv": "Arbitres avec stats (API + Statshub)",
    "data/stades.csv": "Stades avec coordonnées GPS",
    "data/transfermarkt_managers_history.csv": "Historique managers (Transfermarkt)",
}

print("📂 Fichiers générés :")
for path, desc in fichiers.items():
    if os.path.exists(path):
        df = pd.read_csv(path)
        print(f"  ✅ {path} — {df.shape[0]} lignes × {df.shape[1]} colonnes — {desc}")
    else:
        print(f"  ❌ {path} — MANQUANT")

📂 Fichiers générés :
  ✅ data/games.csv — 8721 lignes × 10 colonnes — Matchs joués (FBref) avec noms arbitres normalisés
  ✅ data/managers.csv — 1439 lignes × 36 colonnes — Managers (API Bzzoiro)
  ✅ data/meteo.csv — 10225 lignes × 9 colonnes — Météo par match (Open-Meteo)
  ✅ data/referees.csv — 1385 lignes × 5 colonnes — Arbitres avec stats (API + Statshub)
  ✅ data/stades.csv — 159 lignes × 5 colonnes — Stades avec coordonnées GPS
  ✅ data/transfermarkt_managers_history.csv — 8934 lignes × 5 colonnes — Historique managers (Transfermarkt)
